# Customer Churn Prediction
## Telecom Customer Churn Analysis & Prediction

---

**Author:** Data Science Team  
**Date:** June 2026  
**Objective:** Predict customer churn using machine learning techniques

## 1. Import Libraries

First, we import all the necessary Python libraries for data manipulation, visualization, and machine learning.

In [ ]:
# Data Manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report,
                             roc_auc_score, roc_curve)

# Warning handling
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print('All libraries imported successfully!')

## 2. Data Loading

Load the telecom churn dataset. We'll generate synthetic data for demonstration purposes.

In [ ]:
# Generate synthetic telecom churn data
np.random.seed(42)
n_samples = 5000

data = {
    'customerID': [f'CUST-{i:05d}' for i in range(1, n_samples + 1)],
    'gender': np.random.choice(['Male', 'Female'], n_samples),
    'SeniorCitizen': np.random.choice([0, 1], n_samples, p=[0.84, 0.16]),
    'Partner': np.random.choice(['Yes', 'No'], n_samples, p=[0.48, 0.52]),
    'Dependents': np.random.choice(['Yes', 'No'], n_samples, p=[0.30, 0.70]),
    'tenure': np.random.randint(1, 73, n_samples),
    'PhoneService': np.random.choice(['Yes', 'No'], n_samples, p=[0.90, 0.10]),
    'InternetService': np.random.choice(['DSL', 'Fiber optic', 'No'], n_samples, p=[0.34, 0.44, 0.22]),
    'Contract': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_samples, p=[0.55, 0.21, 0.24]),
    'PaymentMethod': np.random.choice(['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], n_samples),
    'MonthlyCharges': np.round(np.random.uniform(18.25, 118.75, n_samples), 2),
    'TotalCharges': np.round(np.random.uniform(18.85, 8684.80, n_samples), 2)
}

# Create target variable (churn) with some logic
churn_prob = 0.3 + 0.2 * (data['tenure'] < 12).astype(int) + 0.1 * (data['Contract'] == 'Month-to-month').astype(int)
data['Churn'] = np.random.choice(['Yes', 'No'], n_samples, p=[0.27, 0.73])

df = pd.DataFrame(data)

print(f'Dataset shape: {df.shape}')
print(f'\nFirst 5 rows:')
df.head()

## 3. Exploratory Data Analysis (EDA)

Let's explore the data to understand patterns and relationships.

In [ ]:
# Basic information about the dataset
print('='*60)
print('DATASET INFORMATION')
print('='*60)
print(f'\nShape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')
print(f'\nData Types:\n{df.dtypes}')
print(f'\nMissing Values:\n{df.isnull().sum()}')
print(f'\nBasic Statistics:\n{df.describe()}')

In [ ]:
# Churn distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
churn_counts = df['Churn'].value_counts()
axes[0].pie(churn_counts, labels=churn_counts.index, autopct='%1.1f%%', 
            colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[0].set_title('Churn Distribution', fontsize=14, fontweight='bold')

# Bar chart
sns.countplot(data=df, x='Churn', ax=axes[1], palette=['#2ecc71', '#e74c3c'])
axes[1].set_title('Churn Count', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Churn Status')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f'\nChurn Rate: {(df["Churn"] == "Yes").mean() * 100:.2f}%')

In [ ]:
# Churn by Contract Type
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Contract vs Churn
pd.crosstab(df['Contract'], df['Churn']).plot(kind='bar', ax=axes[0, 0], colormap='RdYlGn_r')
axes[0, 0].set_title('Churn by Contract Type', fontweight='bold')
axes[0, 0].set_xlabel('Contract Type')
axes[0, 0].set_ylabel('Count')
axes[0, 0].tick_params(axis='x', rotation=45)

# Internet Service vs Churn
pd.crosstab(df['InternetService'], df['Churn']).plot(kind='bar', ax=axes[0, 1], colormap='RdYlGn_r')
axes[0, 1].set_title('Churn by Internet Service', fontweight='bold')
axes[0, 1].set_xlabel('Internet Service')
axes[0, 1].set_ylabel('Count')
axes[0, 1].tick_params(axis='x', rotation=45)

# Payment Method vs Churn
pd.crosstab(df['PaymentMethod'], df['Churn']).plot(kind='bar', ax=axes[1, 0], colormap='RdYlGn_r')
axes[1, 0].set_title('Churn by Payment Method', fontweight='bold')
axes[1, 0].set_xlabel('Payment Method')
axes[1, 0].set_ylabel('Count')
axes[1, 0].tick_params(axis='x', rotation=45)

# Gender vs Churn
pd.crosstab(df['gender'], df['Churn']).plot(kind='bar', ax=axes[1, 1], colormap='RdYlGn_r')
axes[1, 1].set_title('Churn by Gender', fontweight='bold')
axes[1, 1].set_xlabel('Gender')
axes[1, 1].set_ylabel('Count')
axes[1, 1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Tenure distribution by Churn
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
df[df['Churn']=='Yes']['tenure'].hist(ax=axes[0], alpha=0.7, label='Churned', color='#e74c3c', bins=30)
df[df['Churn']=='No']['tenure'].hist(ax=axes[0], alpha=0.7, label='Retained', color='#2ecc71', bins=30)
axes[0].set_title('Tenure Distribution by Churn', fontweight='bold')
axes[0].set_xlabel('Tenure (months)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Monthly Charges distribution
df[df['Churn']=='Yes']['MonthlyCharges'].hist(ax=axes[1], alpha=0.7, label='Churned', color='#e74c3c', bins=30)
df[df['Churn']=='No']['MonthlyCharges'].hist(ax=axes[1], alpha=0.7, label='Retained', color='#2ecc71', bins=30)
axes[1].set_title('Monthly Charges Distribution by Churn', fontweight='bold')
axes[1].set_xlabel('Monthly Charges ($)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Data Preprocessing

Prepare the data for machine learning models by encoding categorical variables and scaling features.

In [ ]:
# Create a copy of the dataframe
df_model = df.copy()

# Drop customer ID (not needed for prediction)
df_model = df_model.drop('customerID', axis=1)

# Encode binary categorical variables
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'Churn']
le = LabelEncoder()
for col in binary_cols:
    df_model[col] = le.fit_transform(df_model[col])

# One-Hot Encoding for multi-category variables
multi_cat_cols = ['InternetService', 'Contract', 'PaymentMethod']
df_model = pd.get_dummies(df_model, columns=multi_cat_cols, drop_first=True)

print('Preprocessed data shape:', df_model.shape)
print('\nColumns after preprocessing:')
print(df_model.columns.tolist())
df_model.head()

In [ ]:
# Feature and Target separation
X = df_model.drop('Churn', axis=1)
y = df_model['Churn']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Training set size: {X_train.shape[0]} samples')
print(f'Testing set size: {X_test.shape[0]} samples')
print(f'\nChurn distribution in training set: {y_train.value_counts().to_dict()}')
print(f'Churn distribution in testing set: {y_test.value_counts().to_dict()}')

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('\nFeatures scaled successfully!')

## 5. Model Building

We'll train multiple machine learning models and compare their performance.

In [ ]:
# Initialize models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

# Train and evaluate models
results = {}

print('='*70)
print('MODEL TRAINING AND EVALUATION')
print('='*70)

for name, model in models.items():
    print(f'\nTraining {name}...')
    
    # Train
    model.fit(X_train_scaled, y_train)
    
    # Predict
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)
    
    results[name] = {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'ROC-AUC': roc_auc,
        'y_pred': y_pred,
        'y_prob': y_prob
    }
    
    print(f'\n{name} Results:')
    print(f'  Accuracy:  {accuracy:.4f}')
    print(f'  Precision: {precision:.4f}')
    print(f'  Recall:    {recall:.4f}')
    print(f'  F1-Score:  {f1:.4f}')
    print(f'  ROC-AUC:   {roc_auc:.4f}')

## 6. Model Comparison

Visualize and compare the performance of all models.

In [ ]:
# Create comparison dataframe
comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[m]['Accuracy'] for m in results],
    'Precision': [results[m]['Precision'] for m in results],
    'Recall': [results[m]['Recall'] for m in results],
    'F1-Score': [results[m]['F1-Score'] for m in results],
    'ROC-AUC': [results[m]['ROC-AUC'] for m in results]
})

print('\n' + '='*70)
print('MODEL COMPARISON')
print('='*70)
print(comparison_df.to_string(index=False))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Bar plot comparison
comparison_df.set_index('Model')[['Accuracy', 'Precision', 'Recall', 'F1-Score']].plot(
    kind='bar', ax=axes[0], colormap='viridis'
)
axes[0].set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1)
axes[0].legend(loc='lower right')
axes[0].tick_params(axis='x', rotation=0)

# ROC Curves
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    axes[1].plot(fpr, tpr, label=f"{name} (AUC={res['ROC-AUC']:.3f})")

axes[1].plot([0, 1], [0, 1], 'k--', label='Random Classifier')
axes[1].set_title('ROC Curves', fontsize=14, fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

## 7. Confusion Matrix Analysis

Analyze the confusion matrix for the best performing model.

In [ ]:
# Find the best model based on F1-Score
best_model_name = max(results, key=lambda x: results[x]['F1-Score'])
best_model = models[best_model_name]
best_predictions = results[best_model_name]['y_pred']

print(f'\nBest Model: {best_model_name}')
print(f'\nClassification Report:\n')
print(classification_report(y_test, best_predictions, target_names=['Retained', 'Churned']))

# Confusion Matrix
cm = confusion_matrix(y_test, best_predictions)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Retained', 'Churned'],
            yticklabels=['Retained', 'Churned'])
axes[0].set_title(f'Confusion Matrix - {best_model_name}', fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Normalized Confusion Matrix
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Greens', ax=axes[1],
            xticklabels=['Retained', 'Churned'],
            yticklabels=['Retained', 'Churned'])
axes[1].set_title(f'Normalized Confusion Matrix - {best_model_name}', fontweight='bold')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

## 8. Feature Importance

Understand which features are most influential in predicting churn.

In [ ]:
# Get feature importance from Random Forest
rf_model = models['Random Forest']
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print('\nTop 10 Most Important Features:')
print(feature_importance.head(10).to_string(index=False))

# Plot feature importance
plt.figure(figsize=(10, 8))
top_features = feature_importance.head(15)
sns.barplot(data=top_features, x='Importance', y='Feature', palette='viridis')
plt.title('Top 15 Feature Importances (Random Forest)', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## 9. Conclusions & Recommendations

Based on our analysis, we can draw the following conclusions:

### Key Findings:
1. **Contract Type** is the strongest predictor - month-to-month contracts have significantly higher churn
2. **Tenure** negatively correlates with churn - newer customers are more likely to leave
3. **Monthly Charges** positively correlates with churn - higher charges lead to more churn
4. **Internet Service** type affects churn - fiber optic users have higher churn rates

### Recommendations:
1. Offer incentives for customers to switch to annual contracts
2. Focus retention efforts on customers in their first 12 months
3. Review pricing strategy for high-churn segments
4. Improve customer support for fiber optic users

In [ ]:
# Save the best model
import joblib
import os

# Create models directory
os.makedirs('models', exist_ok=True)

# Save model and scaler
joblib.dump(best_model, 'models/best_churn_model.pkl')
joblib.dump(scaler, 'models/scaler.pkl')

print(f'Best model ({best_model_name}) saved successfully!')
print(f'\nFinal Model Performance:')
print(f'  - Accuracy: {results[best_model_name]["Accuracy"]:.2%}')
print(f'  - Precision: {results[best_model_name]["Precision"]:.2%}')
print(f'  - Recall: {results[best_model_name]["Recall"]:.2%}')
print(f'  - F1-Score: {results[best_model_name]["F1-Score"]:.2%}')